# Data Cleaning
Use prem data for exploratory data analysis

In [10]:
# packages
import duckdb
import pandas as pd
from pathlib import Path
import os

In [7]:
# Create one DuckDB connection for the SQL cells below.
con = duckdb.connect()

def load_season(season) -> pd.DataFrame:
    """Load any four-digit season file and register it as SQL table matches."""
    season = str(season)
    if len(season) != 4 or not season.isdigit():
        raise ValueError("season must be a four-digit value such as 2526")

    season_file = Path(
        "/accounts/masters/gautierep/footy_prediction/footy-prediction"
    ) / "data" / "raw" / "football_data" / f"{season}.csv"

    if not season_file.exists():
        raise FileNotFoundError(f"Could not find {season_file}")

    season_df = pd.read_csv(season_file, encoding="latin-1")

    # Matches the season_df pandas df to a virtual SQL table named matches inside duckdb 
    con.register("matches", season_df)
    return season_df


# Load the default season. Change this to load another 2x2x.csv file.
df_2526 = load_season(2526)

# Test the SQL connection and table registration note that this is using the matches table inside duckdb conneciton 
con.sql("SELECT * FROM matches LIMIT 5")

┌─────────┬────────────┬─────────┬─────────────┬─────────────┬───────┬───────┬─────────┬───────┬───────┬─────────┬───────────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬──────────┬──────────┬────────┬────────┬─────────┬─────────┬─────────┬─────────┬─────────┬─────────┬────────┬─────────┬─────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┬─

In [8]:
# create path to sql script which does all the feature engineering.
sql_file = Path(
    "/accounts/masters/gautierep/footy_prediction/footy-prediction"
) / "feature_extraction.sql"

# reads the entire sql file into python as a text string and executes
con.execute(sql_file.read_text())

# Query the final SQL feature table prematch_sql made by the script
prematch_sql = con.sql("""
    SELECT *
    FROM prematch_sql
    ORDER BY match_id
    LIMIT 10
""").df()

prematch_features = prematch_sql[[
    # Info about match
    "match_id",
    "kickoff",
    "HomeTeam",
    "AwayTeam",
    "FTR",
    "HS",
    "AS",
    # Recent form
    "home_points_last_5",
    "away_points_last_5",
    "home_points_last_10",
    "away_points_last_10",
    # venue strength
    "home_points_home_last_5",
    "away_points_away_last_5",
    "home_goals_for_home_last_5",
    "away_goals_for_away_last_5",
    "home_points_home_last_10",
    "away_points_away_last_10",
    "home_goals_for_home_last_10",
    "away_goals_for_away_last_10",
    "market_home_prob_fair",
    "market_draw_prob_fair",
    "market_away_prob_fair",
    # difference in recent form
    "goals_difference_last_5",
    "points_difference_last_5",
    # extra features
    "home_shots_on_target_for_last_5",
    "away_shots_on_target_for_last_5",
    "home_fouls_for_last_5",
    "away_fouls_for_last_5",
    "home_corners_for_last_5",
    "away_corners_for_last_5",
]]


In [27]:
def upload_features(seasons: list):
    "Upload features to csv files for all seasons in the list"

    # make an empty dataframe to append to
    features_df = pd.DataFrame()

    for season_start in range(seasons[0], seasons[1]):
        season_code = (
            f"{season_start % 100:02d}"
            f"{(season_start + 1) % 100:02d}"
        )

        print(f"Processing season {season_code}...")

        season_file = (
            Path(
                "/accounts/masters/gautierep/footy_prediction/"
                "footy-prediction"
            )
            / "data"
            / "raw"
            / "football_data"
            / f"{season_code}.csv"
        )

        if not season_file.exists():
            raise FileNotFoundError(f"Could not find {season_file}")

        # Load the season data
        load_season(season_code)

        # Execute the feature extraction SQL script
        con.execute(sql_file.read_text())

        # Query the final SQL feature table prematch_sql made by the script
        prematch_sql = con.sql("""
            SELECT *
            FROM prematch_sql
            ORDER BY match_id
        """).df()

        # Append the features to the features_df dataframe
        if features_df.empty:
            features_df = prematch_sql
        else:
            features_df = pd.concat([features_df, prematch_sql], ignore_index=True)


    # correct the match_ids as currently they are not unique across seasons
    features_df['match_id'] = list(range(1, len(features_df) + 1))
    season_name = f'{seasons[0]}' + f'{seasons[1]}'
    # Save the features to a CSV file
    output_file = Path(
        "/accounts/masters/gautierep/footy_prediction/footy-prediction"
    ) / "data" / "features" / f"features_data_{season_name}.csv"
        
    features_df.to_csv(output_file, index=False)

    # Print the size of the saved file 
    size_bytes = os.path.getsize(output_file)
    file_row_count = len(features_df)

    for unit in ['bytes', 'KB', 'MB', 'GB']:
        if size_bytes < 1024.0 or unit == 'GB':
            print(f"Saved '{output_file}' successfully. \nSize: {size_bytes:.2f} {unit}\nRow Count: {file_row_count}")
            break
        size_bytes /= 1024.0

upload_features([19, 27])

Processing season 1920...
Processing season 2021...
Processing season 2122...
Processing season 2223...
Processing season 2324...
Processing season 2425...
Processing season 2526...
Processing season 2627...
Saved '/accounts/masters/gautierep/footy_prediction/footy-prediction/data/features/features_data_1927.csv' successfully. 
Size: 2.52 MB
Row Count: 2690


In [15]:
for i in range(20,23):
    print(i)

20
21
22


# APPENDIX

## Feature-engineering logic

The `build_prematch_features` function creates variables that could have been known immediately before each match. It starts by copying the raw table, cleaning the first column name, and combining the date and time into a single `kickoff` column so every match can be ordered chronologically.

The average bookmaker odds are converted into implied probabilities using `1 / odds`. These probabilities include the bookmaker's margin, so the three values are divided by their total (`overround`) to create normalized market probabilities that sum to one. These are useful baseline features because they summarize the market's pre-match expectations.

The match table is then reshaped into team-history rows. Each match produces one row from the home team's perspective and one from the away team's perspective. This makes it possible to calculate each team's previous points, goals, shots, and shots on target consistently, regardless of whether the team played at home or away.

For each team, the function calculates:

- Rest days since the previous match
- Mean points over the previous 5 and 10 matches
- Mean goals scored over the previous 5 and 10 matches
- Mean goals conceded over the previous 5 and 10 matches
- Mean shots and shots on target over the previous 5 and 10 matches

The important leakage safeguard is `.shift(1)`. It moves the history back by one match before calculating each rolling mean. Therefore, a match's own result is never used to create its predictors. The first match for each team has missing historical features because no earlier information exists.

Finally, the home and away history features are merged back onto the original match rows. Difference features, such as `points_difference_last_5`, compare the recent form of the two teams. The resulting `prematch_2526` dataframe contains both the original target column, `FTR`, and the pre-match predictors for later modelling.

1. Recent form
- overall_points_last_5
- overall_points_last_10
- overall_goals_scored_last_5
- overall_goals_conceded_last_5
- overall_shots_last_5
- overall_shots_on_target_last_5
- overall_corners_last_5

2. Venue strength
- home_points_last_5
- home_points_last_10
- home_goals_scored_last_5
- home_goals_conceded_last_5
- away_points_last_5
- away_points_last_10
- away_goals_scored_last_5
- away_goals_conceded_last_5

3. Matchup differences
- home_team_goals_scored_last_5
- away_team_goals_scored_last_5

# Data validation

Take match_id 100 Tottenham vs Man U 

Manually calc away_points_last_5, home_points_home_last_5

below we see that sums are equivalent

In [16]:
list(prematch_2526.columns)



prematch_2526[prematch_2526["match_id"] == 100][['match_id',
    'kickoff',
    'HomeTeam',
    'AwayTeam',
    'FTR',
    'home_points_last_5',
    'away_points_last_5',
    'home_points_home_last_5',
    'away_points_away_last_5',
    'points_difference_last_5',
    'goals_difference_last_5',
    'market_home_prob_fair',
    'market_draw_prob_fair',
    'market_away_prob_fair']].head(10)

,match_id,kickoff,HomeTeam,AwayTeam,FTR,home_points_last_5,away_points_last_5,home_points_home_last_5,away_points_away_last_5,points_difference_last_5,goals_difference_last_5,market_home_prob_fair,market_draw_prob_fair,market_away_prob_fair
100,100,2025-11-08 12:30:00,Tottenham,Man United,D,1.4,2.0,0.8,1.0,-0.6,-0.8,0.35819,0.265141,0.376669


In [ ]:
temp = df_2526[((df_2526["HomeTeam"] == "Tottenham") | (df_2526["AwayTeam"] == "Tottenham") | 
        (df_2526["HomeTeam"] == "Man United") | (df_2526["AwayTeam"] == "Man United"))]

temp["kickoff"] = pd.to_datetime(
    temp["Date"].astype(str) + " " + temp["Time"].fillna("00:00").astype(str),
    dayfirst=True,
    errors="coerce",
)

temp = temp[(temp["kickoff"] > "2025-08-10") & (temp["kickoff"] < "2025-11-09")]
temp[temp["HomeTeam"] == "Tottenham"][["kickoff", "HomeTeam", "AwayTeam", "FTR"]]

/tmp/ipykernel_2104429/3119902237.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  temp["kickoff"] = pd.to_datetime(


,kickoff,HomeTeam,AwayTeam,FTR
4,2025-08-16 15:00:00,Tottenham,Burnley,H
23,2025-08-30 15:00:00,Tottenham,Bournemouth,A
56,2025-09-27 20:00:00,Tottenham,Wolves,D
77,2025-10-19 14:00:00,Tottenham,Aston Villa,A
95,2025-11-01 17:30:00,Tottenham,Chelsea,A
100,2025-11-08 12:30:00,Tottenham,Man United,D
